In [4]:
%pip install google-api-python-client google-auth-httplib2 google-auth-oauthlib python-dateutil python-dotenv --upgrade google-generativeai


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# Importamos librerías de sistema y configuramos las rutas de los archivos de credenciales
import os

CREDENTIALS_PATH = "credentials.json"

# Archivo donde se guardará el token una vez nos autentiquemos la primera vez (para no tener que iniciar sesión en cada ejecución)
TOKEN_PATH = "token.pickle"

print("Usando credenciales en:", CREDENTIALS_PATH)
print("El token se guardará en:", TOKEN_PATH)


Usando credenciales en: credentials.json
El token se guardará en: token.pickle


In [10]:
from __future__ import print_function
import pickle                     # Para guardar y leer el token de sesión
from datetime import datetime
from dateutil import tz            # Para manejar zonas horarias
from googleapiclient.discovery import build   # Cliente de la API de Google
from google_auth_oauthlib.flow import InstalledAppFlow  # Para el login OAuth
from google.auth.transport.requests import Request      # Para refrescar credenciales

# Alcance (scope) de los permisos que pedimos: en este caso, acceso completo al calendario
SCOPES = ['https://www.googleapis.com/auth/calendar']

def get_service():
    """
    Esta función:
    1. Comprueba si ya tenemos un token guardado (para no volver a loguear).
    2. Si no hay token válido, abre una ventana de login en el navegador.
    3. Devuelve un objeto 'service' que nos permite interactuar con la API de Google Calendar.
    """

    creds = None

    # Si ya existe un token guardado, lo cargamos
    if os.path.exists(TOKEN_PATH):
        with open(TOKEN_PATH, 'rb') as f:
            creds = pickle.load(f)

    # Si no hay credenciales válidas, pedimos login al usuario
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            # Si el token está caducado pero tiene refresh_token, se renueva automáticamente
            creds.refresh(Request())
        else:
            # Si no hay token, iniciamos el flujo de autorización OAuth
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_PATH, SCOPES)
            creds = flow.run_local_server(port=0)  # Abre navegador para login con tu cuenta de Google

        # Guardamos el token en disco para la próxima vez
        with open(TOKEN_PATH, 'wb') as f:
            pickle.dump(creds, f)

    # Creamos el objeto 'service' para trabajar con la API de Calendar
    return build('calendar', 'v3', credentials=creds)

print("Funciones cargadas. Listo para autenticar.")


Funciones cargadas. Listo para autenticar.


In [11]:
# Obtenemos el servicio autenticado
svc = get_service()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=11980723519-frkqr1ebvndrvjpsof08rkrrhsjca36b.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A61973%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcalendar&state=r6oLP6SjGeYsV7YSxi6u8Cmh7LAZrA&access_type=offline


In [85]:


# # Definimos un evento simple con título, hora de inicio y fin, descripción y localización
# event = {
#     'summary': 'Reunión de prueba',  
#     'colorId':8,  # Título del evento
#     'description': 'Prueba de la API de Google Calendar desde VS Code', 
#     'location': 'Online',              # Lugar (opcional)
#     'start': {
#         'dateTime': '2025-10-05T10:00:00+02:00',  # Fecha/hora inicio en formato ISO8601
#         'timeZone': 'Europe/Madrid'               # Zona horaria
#     },
#     'end': {
#         'dateTime': '2025-10-05T11:00:00+02:00',  # Fecha/hora fin
#         'timeZone': 'Europe/Madrid'
#     }
    
# }

# # Insertamos el evento en el calendario principal (primary)
# created_event = svc.events().insert(calendarId='primary', body=event).execute()

# # Mostramos el enlace al evento en Google Calendar
# print("Evento creado ✅:", created_event.get('htmlLink'))


Evento creado ✅: https://www.google.com/calendar/event?eid=MjJjZ3Noc2IwaGdqajhmYWFlODdpbmY3aDAgcmVpbmFyb21lcm9jbGFyYUBt


In [ ]:
# # Mostramos los próximos 5 eventos en tu calendario principal
# from datetime import datetime, timezone

# events_result = svc.events().list(
#     calendarId='primary',      # 'primary' = tu calendario principal
#     timeMin = datetime.now(timezone.utc).isoformat(),   # a partir de la fecha actual
#     maxResults=5,              # número de eventos a listar
#     singleEvents=True,         # expande eventos recurrentes
#     orderBy='startTime'        # ordena por fecha de inicio
# ).execute()

# events = events_result.get('items', [])

# if not events:
#     print("No se encontraron próximos eventos.")
# else:
#     print("Próximos eventos:")
#     for e in events:
#         start = e['start'].get('dateTime', e['start'].get('date'))  # puede ser con hora o todo el día
#         print(f"- {start} | {e['summary']} | id={e['id']}")


Próximos eventos:
- 2025-10-06T12:00:00+02:00 | Tutoría con Pedro | id=0drj91n9rhds77khrehp9eqmpk_20251006T100000Z
- 2025-10-06T12:00:00+02:00 | Tutoría con Pedro | id=f48lhcsnt9rpofncgm5rckbvfk
- 2025-10-07 | cumple pilar | id=cdh3gohhcpgj4bb4c5hm2b9k71h3abb16ko3ebb5cdgj6p9m60o36ohl60_20251007
- 2025-10-10T09:00:00+02:00 | Cita médica | id=ak81og87gmqjf7cnpf0n29c4b4
- 2025-10-10T10:00:00+02:00 | Reunión final | id=8npt4n2f90pvl0pvltqblkor7g


In [ ]:
# #Muestra eventos de esta semana
# from datetime import datetime, timedelta, timezone

# # Ahora mismo en UTC
# now = datetime.now(timezone.utc)

# # Primer día del rango: hoy (desde este momento)
# time_min = now.isoformat()

# # Último día del rango: dentro de 7 días
# time_max = (now + timedelta(days=7)).isoformat()

# # Pedimos a la API todos los eventos entre hoy y dentro de 7 días
# events_result = svc.events().list(
#     calendarId='primary',
#     timeMin=time_min,
#     timeMax=time_max,
#     singleEvents=True,
#     orderBy='startTime'
# ).execute()

# events = events_result.get('items', [])

# if not events:
#     print("No se encontraron eventos en esta semana.")
# else:
#     print("Eventos de esta semana:")
#     for e in events:
#         start = e['start'].get('dateTime', e['start'].get('date'))
#         end = e['end'].get('dateTime', e['end'].get('date'))
#         print(f"- {start} → {end} | {e['summary']} | id={e['id']}")


Eventos de esta semana:
- 2025-10-07 → 2025-10-08 | cumple pilar | id=cdh3gohhcpgj4bb4c5hm2b9k71h3abb16ko3ebb5cdgj6p9m60o36ohl60_20251007


In [39]:
# #Modificar un evento con PATCH
# event_id = "i74vg2okdftup5teds1c7ku6q0"

# # Cambiamos título, descripción y ubicación de una sola vez
# patched_event = svc.events().patch(
#     calendarId='primary',
#     eventId=event_id,
#     body={
#         "summary": "Reunión editada con PATCH",
#         "description": "Ahora he cambiado varios campos de golpe.",
#         "location": "Sala de reuniones 3"
#     }
# ).execute()

# print("Evento actualizado con PATCH ✅:", patched_event.get('htmlLink'))


In [40]:
# #Modificar un evento con PUT/UPDATE

# event_id = "i74vg2okdftup5teds1c7ku6q0"

# # Obtenemos el evento actual completo
# event = svc.events().get(calendarId='primary', eventId=event_id).execute()

# print("Título original:", event['summary'])

# # Modificamos los campos que nos interesen
# event['summary'] = "Reunión editada con UPDATE"
# event['description'] = "He cambiado el título, la descripción y la ubicación."
# event['location'] = "Sala de reuniones 2"

# # 🔎 IMPORTANTE: si quieres cambiar fechas/horas también,
# # modifica los campos start y end, por ejemplo:
# # event['start']['dateTime'] = "2025-10-01T09:00:00+02:00"
# # event['end']['dateTime']   = "2025-10-01T10:00:00+02:00"

# # Enviamos el evento completo de vuelta a Google Calendar
# updated_event = svc.events().update(
#     calendarId='primary',
#     eventId=event_id,
#     body=event
# ).execute()

# print("Evento actualizado con UPDATE ✅:", updated_event.get('htmlLink'))


In [41]:
# #CREAMOS OTRO EVENTO PARA ELIMINAR DESPUÉS
# # Definimos un evento simple con título, hora de inicio y fin, descripción y localización
# event = {
#     'summary': 'Reunión de prueba',    # Título del evento
#     'description': 'Prueba de la API de Google Calendar desde VS Code', 
#     'location': 'Online',              # Lugar (opcional)
#     'start': {
#         'dateTime': '2025-09-30T12:00:00+02:00',  # Fecha/hora inicio en formato ISO8601
#         'timeZone': 'Europe/Madrid'               # Zona horaria
#     },
#     'end': {
#         'dateTime': '2025-09-30T15:00:00+02:00',  # Fecha/hora fin
#         'timeZone': 'Europe/Madrid'
#     }
# }

# # Insertamos el evento en el calendario principal (primary)
# created_event = svc.events().insert(calendarId='primary', body=event).execute()

# # Mostramos el enlace al evento en Google Calendar
# print("Evento creado ✅:", created_event.get('htmlLink'))
# print("ID evento:", created_event.get("id"))


In [42]:
# #Eliminar evento

# event_id = "8kv970rbmi3hnu22d7bulkadbg"

# # 2. Llamamos al método delete
# svc.events().delete(
#     calendarId='primary',   # tu calendario principal
#     eventId=event_id
# ).execute()

# print("Evento eliminado ✅")


In [ ]:
# from datetime import datetime, timedelta, timezone

# svc = get_service()

# # Fecha actual
# ahora = datetime.now(timezone.utc)

# # Inicio: 1 mes antes
# inicio = (ahora - timedelta(days=30)).isoformat()

# # Fin: 1 año después
# fin = (ahora + timedelta(days=365)).isoformat()

# # Recuperar eventos en ese rango
# events_result_general = svc.events().list(
#     calendarId="primary",
#     timeMin=inicio,
#     timeMax=fin,
#     maxResults=2500,          # límite máximo que permite la API
#     singleEvents=True,
#     orderBy="startTime"
# ).execute()

# events_general = events_result_general.get("items", [])

# print(f"Eventos encontrados en el rango {inicio} → {fin}: {len(events_general)}")
# for e in events_general:
#     start = e['start'].get('dateTime', e['start'].get('date'))
#     print(f"- {start} | {e['summary']} | id={e['id']}")


Eventos encontrados en el rango 2025-09-04T14:26:55.944252+00:00 → 2026-10-04T14:26:55.944252+00:00: 70
- 2025-09-05 | cumple prima marta | id=_7114cdho64qj8ba48csjeb9k8l1j4ba184s30b9h84rjacpp8gs34dq274_20250905
- 2025-09-12 | Cumple Ángela | id=7jugqt7e40sgbp6g04q1gljteq_20250912
- 2025-09-14 | cumple abuelo pepe | id=6srjacj360p38bb6c5h36b9k70sjgbb260q62b9ocph6aeb3cgqjedhh6s_20250914
- 2025-09-14 | cumple helena | id=chh3edhl6co66b9nc9hjib9k6hij0b9o74o3cb9l6cq36e36c4s3echg60_20250914
- 2025-09-15 | Cumple Alejandra | id=41jh9cvprqqvu693enbpit94o7_20250915
- 2025-09-15 | Cumple Mamá | id=6orjecr2c4r32b9j71j6ab9k71h3gb9o74o34b9mcoom4e1gcgrjgeb674_20250915
- 2025-09-15 | Cumple Arancha | id=70oearvou46kcgp9nhuqf9o7tv_20250915
- 2025-09-24 | cumple candela | id=c8qjgdj3cco68b9j6kojeb9k6cs38bb1cphm2b9p6hh3ephnc8o36o9j6s_20250924
- 2025-09-27 | Cumple Nico | id=cdj62opo64r32bb6cco34b9k6dimab9p6th68bb6ccoj6pb66orm8dplc4_20250927
- 2025-10-03 | Cumpleaños Clara Fernández Potter | id=6lij4ohl

In [29]:
from datetime import datetime, timedelta
from dateutil import tz
import uuid

def create_event(
    service = svc, name = None,
    start_date = None,
    end_date = None, start_time=None, end_time=None,
    description="",
    color_id="7",                 # Azul
    visibility="default",         # "default" | "public" | "private"
    transparency="opaque",        # "opaque" (ocupa) | "transparent" (no bloquea)
    location="",
    attendees=None,                  # lista de emails
    default_reminder = True,
    reminder=None,                # lista de dicts [{"method":"popup","minutes":10}]
    zone="Europe/Madrid",
    recurrence=None,              # lista de strings tipo ["RRULE:..."]
    calendar_id="primary",
    attachments=None,             # lista de dicts [{"fileUrl":"...","title":"..."}]
    conference = False,            
    source=None,                  # dict {"url":"...","title":"..."}
    send_updates="none"           # "none" | "all" | "externalOnly"

):
      
    if not start_date: start_date = datetime.now().date().isoformat()
    
    event = {"summary": name}

    #----------CAMPOS DE FECHA Y HORA----------

    #si dice fecha y hora inicio y fin
    if end_date and start_time and end_time:
        start_dt = f"{start_date}T{start_time}:00"
        end_dt   = f"{end_date}T{end_time}:00"
        event["start"] = {"dateTime": start_dt, "timeZone": zone}
        event["end"]   = {"dateTime": end_dt,   "timeZone": zone}

    #si dice fecha y hora inicio (sin fin) -> 1h por defecto
    elif start_time and not end_time and not end_date:
        # Construir datetime con fecha y hora de inicio
        inicio = datetime.strptime(f"{start_date} {start_time}", "%Y-%m-%d %H:%M")
        fin = inicio + timedelta(hours=1)

        # Convertir a string ISO para Calendar
        start_dt = inicio.isoformat()
        end_dt   = fin.isoformat()

        event["start"] = {"dateTime": start_dt, "timeZone": zone}
        event["end"]   = {"dateTime": end_dt,   "timeZone": zone}


    #si dice fecha inicio y fin sin hora (varios dia completo)
    elif end_date and not start_time and not end_time:
        event["start"] = {"date": start_date}
        event["end"]   = {"date": end_date}

    #si dice fecha inicio sin hora (1 dia completo)
    else:
        end_date = (datetime.fromisoformat(start_date) + timedelta(days=1)).date().isoformat()
        event["start"] = {"date": start_date}
        event["end"]   = {"date": end_date}


    #----------CAMPOS DE CONTENIDO----------

    if description: event["description"] = description
    if location:   event["location"]    = location
    if color_id:    event["colorId"]     = str(color_id)    
    if visibility: event["visibility"]  = visibility      
    if transparency: event["transparency"] = transparency


    #--------------------
    if recurrence: event["recurrence"] = recurrence   # Ejemplo: ["RRULE:FREQ=WEEKLY;BYDAY=MO", "EXDATE:20251006T100000Z"]

    if attendees: event["attendees"] = [{"email": m} for m in attendees]

    if default_reminder:
        event["reminders"] = {"useDefault": True}
    else:
        event["reminders"] = {
            "useDefault": False,
            "overrides": reminder or []  # [{"method":"popup","minutes":10}, {"method":"email","minutes":1440}]
        }
    
    if attachments: event["attachments"] = attachments

    if conference: event["conferenceData"] = {
            "createRequest": {
                "requestId": str(uuid.uuid4()),              # id único por petición
                "conferenceSolutionKey": {"type": "hangoutsMeet"}
            }
        }
    
    if source: event["source"] = source

# ---------- 9) Llamada a la API ----------
    # Param extras: sendUpdates (emails a invitados) y supportsAttachments
    params_insert = {
        "calendarId": calendar_id,
        "body": event
    }
    if conference:
        params_insert["conferenceDataVersion"] = 1
    if attachments:
        params_insert["supportsAttachments"] = True
    # Envío de correos a invitados
    params_insert["sendUpdates"] = send_updates  # "none" (por defecto), "all", "externalOnly"

    created = service.events().insert(**params_insert).execute()

    print("✅ Evento creado:", created.get("htmlLink"))
    if created.get("hangoutLink"):
        print("🔗 Meet:", created["hangoutLink"])
    print("🆔 ID:", created["id"])
    return created

In [47]:
from datetime import datetime, timedelta, timezone

start_date = datetime.now(timezone.utc).isoformat()
end_date = (datetime.now(timezone.utc) + timedelta(days=365)).isoformat()

print(start_date)
print(end_date)

2025-10-04T14:26:56.738067+00:00
2026-10-04T14:26:56.738067+00:00


In [30]:
def get_id(name, start_date = None, end_date = None, calendar_id = "primary", service = svc):
    if not start_date: timeMin = datetime.now(timezone.utc).isoformat()
    if not end_date: timeMax = (datetime.now(timezone.utc) + timedelta(days=365)).isoformat()
    # Recuperar eventos en ese rango
    events_result = svc.events().list(
        calendarId=calendar_id,
        timeMin=timeMin,
        timeMax=timeMax,
        maxResults=2500,          # límite máximo que permite la API
        singleEvents=True,
        orderBy="startTime"
    ).execute()

    events = events_result.get("items", [])

    for e in events:
        if start_date:
            if e["summary"] == name and e["start"] == start_date:
                return e["id"]
        else:
            if e["summary"] == name:
                return e["id"]
    print("No se encontró el evento")


In [31]:
def delete_event(name, start_date = None, end_date = None, 
                    calendar_id = "primary", service = svc):
    event_id = get_id(name, start_date, end_date, calendar_id, service)
    if event_id:
        svc.events().delete(
            calendarId=calendar_id,
            eventId=event_id
        ).execute()

        return "Evento eliminado ✅"
    else:
        return None

In [32]:
def get_events(name =  None,
        start_date = None, end_date = None, calendar_id = "primary",
        service = svc, max = 2500
):
    #por defecto una semana
    if not start_date: start_date = datetime.now(timezone.utc).isoformat()
    if not end_date: end_date = (datetime.now(timezone.utc) + timedelta(days=30)).isoformat()
    
    #busqueda por nombre
    if name:
         events_result = svc.events().list(
        calendarId=calendar_id,
        timeMin=start_date,
        timeMax=end_date,
        q=name,
        maxResults = max,
        singleEvents=True,
        orderBy='startTime'
        ).execute()
         
    else:   
        events_result = svc.events().list(
            calendarId=calendar_id,
            timeMin=start_date,
            timeMax=end_date,
            maxResults = max,
            singleEvents=True,
            orderBy='startTime'
        ).execute()
    
    events = events_result.get('items', [])

    if not events:
        print("No se encontraron próximos eventos.")
    else:
        print("Próximos eventos:")
        for e in events:
            start = e['start'].get('dateTime', e['start'].get('date'))  # puede ser con hora o todo el día
            print(f"- {start} | {e['summary']} | id={e['id']}")
    return events

In [33]:
def patch_event(name, start_date = None, changes = None, service = svc):
    # changes puede ser {"summary": "nuevo título", "description": "texto"}
    # o {"start": {...}, "end": {...}}

    #Modificar un evento con PATCH
    event_id = get_id(name = name, start_date = start_date)

    # Cambiamos título, descripción y ubicación de una sola vez
    patched_event = svc.events().patch(
        calendarId='primary',
        eventId=event_id,
        body=changes
    ).execute()

    print("Evento actualizado con PATCH ✅:",
           patched_event.get('htmlLink'))

In [34]:
def duplicate_event(
    service=svc,
    name=None,
    original_date=None,
    new_date=None,
    new_time=None,
    calendar_id="primary"
):
    if not name or not new_date:
        print("Debes indicar el nombre del evento original y la nueva fecha")
        return None

    # Buscar evento original
    eventos = get_events(name=name, start_date=original_date,
                          calendar_id=calendar_id, service=service)

    if not eventos:
        print("No se encontró el evento para duplicar")
        return None

    original = eventos[0]
    if not new_time: 
        new_time = original["start"].get("dateTime", original["start"]
                                         .get("date", "00:00"))[11:16]


    # Crear el duplicado copiando directamente los datos
    nuevo = create_event(
        service=service,
        name=original.get("summary", "(sin título)"),
        start_date=new_date,
        start_time=new_time,
        description=original.get("description", ""),
        location=original.get("location", ""),
        color_id=original.get("colorId", "1"),
        attendees=[a["email"] for a in original.get("attendees", [])],
        recurrence=original.get("recurrence"),
        attachments=original.get("attachments"),
        source=original.get("source"),
        calendar_id=calendar_id
    )

    print(f"Evento duplicado: {nuevo.get('htmlLink')}")
    return nuevo


In [ ]:
# #EJEMPLOS

# #Ejemplo básico
# create_event(
#     service=svc,
#     name="Reunión de equipo",
#     start_date="2025-10-10",
#     end_date="2025-10-10",
#     start_time="10:00",
#     end_time="11:00",
#     description="Seguimiento semanal del proyecto TFG",
#     location="Sala 2 - Facultad",
#     color_id="7",             # Azul (default)
#     visibility="default",     # visible según config del calendario
#     transparency="opaque",    # bloquea tiempo
#     calendar_id="primary"
# )


✅ Evento creado: https://www.google.com/calendar/event?eid=OG5wdDRuMmY5MHB2bDBwdmx0cWJsa29yN2cgcmVpbmFyb21lcm9jbGFyYUBt
🆔 ID: 8npt4n2f90pvl0pvltqblkor7g


{'kind': 'calendar#event',
 'etag': '"3519176032940478"',
 'id': '8npt4n2f90pvl0pvltqblkor7g',
 'status': 'confirmed',
 'htmlLink': 'https://www.google.com/calendar/event?eid=OG5wdDRuMmY5MHB2bDBwdmx0cWJsa29yN2cgcmVpbmFyb21lcm9jbGFyYUBt',
 'created': '2025-10-04T14:26:56.000Z',
 'updated': '2025-10-04T14:26:56.470Z',
 'summary': 'Reunión de equipo',
 'description': 'Seguimiento semanal del proyecto TFG',
 'location': 'Sala 2 - Facultad',
 'colorId': '7',
 'creator': {'email': 'reinaromeroclara@gmail.com', 'self': True},
 'organizer': {'email': 'reinaromeroclara@gmail.com', 'self': True},
 'start': {'dateTime': '2025-10-10T10:00:00+02:00',
  'timeZone': 'Europe/Madrid'},
 'end': {'dateTime': '2025-10-10T11:00:00+02:00', 'timeZone': 'Europe/Madrid'},
 'iCalUID': '8npt4n2f90pvl0pvltqblkor7g@google.com',
 'sequence': 0,
 'reminders': {'useDefault': True},
 'eventType': 'default'}

In [ ]:
# get_id(name="Reunión de prueba")

No se encontró el evento


In [ ]:
# delete_event(name="Reunión de prueba")

No se encontró el evento


In [ ]:
# get_events()

Próximos eventos:
- 2025-10-07 | cumple pilar | id=cdh3gohhcpgj4bb4c5hm2b9k71h3abb16ko3ebb5cdgj6p9m60o36ohl60_20251007
- 2025-10-10T10:00:00+02:00 | Reunión de equipo | id=8npt4n2f90pvl0pvltqblkor7g
- 2025-10-12 | Cumple Pepe Grande  | id=64sjcphk6tgj6b9pcli6cb9k6go3ibb1cdi68b9j6ph38c9p6srj6eb670_20251012
- 2025-10-19 | cumple migueeel | id=6gr38e326soj8bb3c8r36b9k69j3gb9ochj3ib9nc4sjecj46hgm2cr66o_20251019
- 2025-10-26 | cumple blanca | id=coojco9i68p34b9i6cpm2b9k6orjibb26gp32bb3ccs36p35cpij4cr560_20251026


[{'kind': 'calendar#event',
  'etag': '"3267225806174000"',
  'id': 'cdh3gohhcpgj4bb4c5hm2b9k71h3abb16ko3ebb5cdgj6p9m60o36ohl60_20251007',
  'status': 'confirmed',
  'htmlLink': 'https://www.google.com/calendar/event?eid=Y2RoM2dvaGhjcGdqNGJiNGM1aG0yYjlrNzFoM2FiYjE2a28zZWJiNWNkZ2o2cDltNjBvMzZvaGw2MF8yMDI1MTAwNyByZWluYXJvbWVyb2NsYXJhQG0',
  'created': '2021-10-07T13:21:43.000Z',
  'updated': '2021-10-07T13:21:43.087Z',
  'summary': 'cumple pilar',
  'colorId': '10',
  'creator': {'email': 'reinaromeroclara@gmail.com', 'self': True},
  'organizer': {'email': 'reinaromeroclara@gmail.com', 'self': True},
  'start': {'date': '2025-10-07'},
  'end': {'date': '2025-10-08'},
  'recurringEventId': 'cdh3gohhcpgj4bb4c5hm2b9k71h3abb16ko3ebb5cdgj6p9m60o36ohl60',
  'originalStartTime': {'date': '2025-10-07'},
  'transparency': 'transparent',
  'iCalUID': 'cdh3gohhcpgj4bb4c5hm2b9k71h3abb16ko3ebb5cdgj6p9m60o36ohl60@google.com',
  'sequence': 0,
  'reminders': {'useDefault': False,
   'overrides': [{'me

In [ ]:
# get_id(name="Reunión de equipo")

'8npt4n2f90pvl0pvltqblkor7g'

In [ ]:
# patch_event(
#     service=svc,
#     name="Reunión de equipo",      # el nombre del evento a buscar
#     # fecha="2025-10-05",            # opcional, restringe al día
#     changes={                      # los cambios que quieres aplicar
#         "summary": "Reunión final",
#         "description": "Actualizada automáticamente"
#     }
# )


Evento actualizado con PATCH ✅: https://www.google.com/calendar/event?eid=OG5wdDRuMmY5MHB2bDBwdmx0cWJsa29yN2cgcmVpbmFyb21lcm9jbGFyYUBt


In [ ]:
# get_events(name="Reunión final")

Próximos eventos:
- 2025-10-10T10:00:00+02:00 | Reunión final | id=8npt4n2f90pvl0pvltqblkor7g


[{'kind': 'calendar#event',
  'etag': '"3519176035982302"',
  'id': '8npt4n2f90pvl0pvltqblkor7g',
  'status': 'confirmed',
  'htmlLink': 'https://www.google.com/calendar/event?eid=OG5wdDRuMmY5MHB2bDBwdmx0cWJsa29yN2cgcmVpbmFyb21lcm9jbGFyYUBt',
  'created': '2025-10-04T14:26:56.000Z',
  'updated': '2025-10-04T14:26:57.991Z',
  'summary': 'Reunión final',
  'description': 'Actualizada automáticamente',
  'location': 'Sala 2 - Facultad',
  'colorId': '7',
  'creator': {'email': 'reinaromeroclara@gmail.com', 'self': True},
  'organizer': {'email': 'reinaromeroclara@gmail.com', 'self': True},
  'start': {'dateTime': '2025-10-10T10:00:00+02:00',
   'timeZone': 'Europe/Madrid'},
  'end': {'dateTime': '2025-10-10T11:00:00+02:00',
   'timeZone': 'Europe/Madrid'},
  'iCalUID': '8npt4n2f90pvl0pvltqblkor7g@google.com',
  'sequence': 0,
  'reminders': {'useDefault': True},
  'eventType': 'default'}]

In [ ]:
# duplicate_event(
#     service=svc,
#     name="Reunión final",     # nombre del evento original
#     new_date = "2025-10-11" # fecha para el duplicado
# )


Próximos eventos:
- 2025-10-10T10:00:00+02:00 | Reunión final | id=8npt4n2f90pvl0pvltqblkor7g
✅ Evento creado: https://www.google.com/calendar/event?eid=bjhnNmZzMzZnMjg4aGx1ZXIyOGJvNTkxcjAgcmVpbmFyb21lcm9jbGFyYUBt
🆔 ID: n8g6fs36g288hluer28bo591r0
Evento duplicado: https://www.google.com/calendar/event?eid=bjhnNmZzMzZnMjg4aGx1ZXIyOGJvNTkxcjAgcmVpbmFyb21lcm9jbGFyYUBt


{'kind': 'calendar#event',
 'etag': '"3519176037336894"',
 'id': 'n8g6fs36g288hluer28bo591r0',
 'status': 'confirmed',
 'htmlLink': 'https://www.google.com/calendar/event?eid=bjhnNmZzMzZnMjg4aGx1ZXIyOGJvNTkxcjAgcmVpbmFyb21lcm9jbGFyYUBt',
 'created': '2025-10-04T14:26:58.000Z',
 'updated': '2025-10-04T14:26:58.668Z',
 'summary': 'Reunión final',
 'description': 'Actualizada automáticamente',
 'location': 'Sala 2 - Facultad',
 'colorId': '7',
 'creator': {'email': 'reinaromeroclara@gmail.com', 'self': True},
 'organizer': {'email': 'reinaromeroclara@gmail.com', 'self': True},
 'start': {'dateTime': '2025-10-11T10:00:00+02:00',
  'timeZone': 'Europe/Madrid'},
 'end': {'dateTime': '2025-10-11T11:00:00+02:00', 'timeZone': 'Europe/Madrid'},
 'iCalUID': 'n8g6fs36g288hluer28bo591r0@google.com',
 'sequence': 0,
 'reminders': {'useDefault': True},
 'eventType': 'default'}

In [14]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

# Listar modelos disponibles
# for m in genai.list_models():
#     print(m.name, "→", m.supported_generation_methods)

model = genai.GenerativeModel("models/gemini-2.5-flash")
response = model.generate_content("Hola Gemini")
print(response.text)


c:\TFG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


¡Hola! ¿En qué puedo ayudarte hoy?


In [15]:
from datetime import datetime
import json

# Obtener la fecha actual del sistema en formato YYYY-MM-DD
fecha_actual = datetime.now().strftime("%Y-%m-%d")

p = """
Eres un traductor de lenguaje natural a funciones Python de un agente de calendario.

Formato de salida:
{
  "function": "<create_event | delete_event | duplicate_event | patch_event | get_events>",
  "parameters": { ... }
}
Si hay varias instrucciones (1., 2., 3., …), devuelve un ARRAY JSON con un objeto por instrucción, en el mismo orden.

────────────────────────────────────────
Funciones disponibles y explicación

1) create_event
Crea un evento en Google Calendar.
Parámetros: name, start_date, end_date, start_time, end_time, description, location, attendees, color_id, recurrence, attachments, conference, source, default_reminder, reminder, zone, calendar_id, send_updates, visibility, transparency.

2) delete_event
Elimina un evento existente en el calendario.
Parámetros: name, start_date, end_date (opcional para rangos).
Llama a get_id para obtener el eventId.

3) duplicate_event
Duplica un evento existente en otra fecha (y opcionalmente otra hora).
Parámetros: name, original_date, new_date, new_time (opcional).

4) patch_event
Modifica parcialmente un evento (título, hora, ubicación, color, recordatorios, asistentes, recurrencia…).
Parámetros: name, start_date, changes (objeto con los campos a modificar, p. ej. summary, description, location, colorId, attendees, recurrence, reminders, start/end con date/time separados).
Llama a get_id para obtener el eventId.

5) get_events
Obtiene los eventos de un periodo o filtrados por nombre.
Parámetros: name (opcional), start_date (opcional), end_date (opcional), calendar_id (opcional, por defecto "primary"), max (opcional, por defecto 2500).
Si no se indica rango de fechas, se obtienen los próximos 30 días.

────────────────────────────────────────
Reglas de interpretación (OBLIGATORIAS)

A) Invitados (attendees)
- Si el usuario dice “invita a X” pero NO proporciona emails válidos, NO inventes correos.
- En ese caso, pon "attendees": [] (lista vacía) o simplemente omite el campo "attendees".
- Solo rellena "attendees" cuando el usuario dé emails reales (con @).

B) Colores (color_id) — usa ÚNICAMENTE estos IDs
Por defecto el color será el del calendario, o el azul con id 7.
Mapea el color mencionado a uno de estos IDs. Si no coincide, no pongas color_id.
- rojo → "11" (sinónimos: rojo, tomate)
- azul arándano → "9" (sinónimos: azul arándano, azul oscuro)
- amarillo → "5" (sinónimos: amarillo, amarillo huevo)
- verde oscuro → "10" (sinónimos: verde, verde musgo, verde oscuro)
- naranja → "6" (sinónimos: naranja, mandarina)
- morado/violeta → "3" (sinónimos: morado, violeta, púrpura)
- azul (turquesa) → "7" (sinónimos: turquesa, cian, azul, azul turquesa, azul claro)
- gris → "8" (sinónimos: gris, plomo)
- rosa → "4" (sinónimos: rosa, flamenco, coral, rosa chicle)
- lavanda/lila claro → "1" (sinónimos: lavanda, lila claro, morado claro)
- verde claro → "2" (sinónimos: verde claro, salvia, verde esmeralda)
Si el usuario pide "color X" y X no está en la lista, omite color_id.

C) Duración y fechas
- Las fechas y horas se expresan SIEMPRE en campos separados:
  - start_date, end_date → "YYYY-MM-DD"
  - start_time, end_time → "HH:MM"
- "Todo el día" en UN SOLO DÍA → solo start_date (sin end_date ni horas).
- "Todo el día" en VARIOS DÍAS → start_date y end_date (sin horas).
- Evento con horas:
  - Si hay start_time y NO hay end_time → dura 1 hora por defecto.
  - Si hay start_time y end_time → se usan ambas en el mismo día.
- No inventes horas ni fechas si el usuario no las dice.
- Si no dice hora ni "todo el día", se considera evento de día completo.
- Los campos de tiempo usan formato 24h, sin segundos ni zona horaria.

D) Verbos y sinónimos → función
- create_event ↔ crea, agenda, organiza, programa, pon, añade, agrega.
- delete_event ↔ elimina, borra, quita, suprime, cancela.
- duplicate_event ↔ duplica, copia, clona, repite, replica.
- patch_event ↔ cambia, modifica, ajusta, edita, actualiza.
- get_events ↔ dime, muéstrame, enséñame, lista, muestra, consulta, busca, obtén.

E) Formatos
- Fecha: "YYYY-MM-DD"
- Hora: "HH:MM"
- No se incluyen segundos ni zona horaria.
- Todas las fechas y horas deben seguir este formato exacto.

────────────────────────────────────────
Ejemplos (casuísticas clave)
# 1) create_event — un día "todo el día"
Usuario: "Bloquea estudio el 10 de octubre todo el día"
Respuesta:
{ 
  "function": "create_event",
  "parameters": {
    "name": "Bloqueo de estudio",
    "start_date": "2025-10-10"
  }
}

# 2) create_event — varios días "todo el día"
Usuario: "Viaje de 10 a 12 de octubre todo el día"
Respuesta:
{
  "function": "create_event",
  "parameters": {
    "name": "Viaje",
    "start_date": "2025-10-10",
    "end_date": "2025-10-12"
  }
}

# 3) create_event — con horas
Usuario: "Cita médica el 7 de octubre de 09:30 a 10:15"
Respuesta:
{
  "function": "create_event",
  "parameters": {
    "name": "Cita médica",
    "start_date": "2025-10-07",
    "start_time": "09:30",
    "end_date": "2025-10-07",
    "end_time": "10:15"
  }
}

# 4) create_event — con hora única
Usuario: "Reunión con Ana mañana a las 10 en la oficina"
#(en el ejemplo, hoy es 2025-10-03)
Respuesta:
{
  "function": "create_event",
  "parameters": {
    "name": "Reunión con Ana",
    "start_date": "2025-10-04",
    "start_time": "10:00",
    "location": "Oficina"
  }
}

# 5) create_event — invitar sin correos
Usuario: "Pon reunión con Laura mañana a las 9 e invita a Laura"
Respuesta:
{
  "function": "create_event",
  "parameters": {
    "name": "Reunión con Laura",
    "start_date": "2025-10-04",
    "start_time": "09:00",
    "attendees": []
  }
}

# 6) create_event — invitar con correos
Usuario: "Reunión de equipo el jueves a las 16h con pedro@gmail.com y marta@gmail.com"
#(en el ejemplo, hoy es 2025-10-06)
Respuesta:
{
  "function": "create_event",
  "parameters": {
    "name": "Reunión de equipo",
    "start_date": "2025-10-09",
    "start_time": "16:00",
    "attendees": ["pedro@gmail.com","marta@gmail.com"]
  }
}

# 7) create_event — color correcto
Usuario: "Marca la reunión de ventas en rojo el 12 de octubre a las 11"
Respuesta:
{
  "function": "create_event",
  "parameters": {
    "name": "Reunión de ventas",
    "start_date": "2025-10-12",
    "start_time": "11:00",
    "color_id": "11"
  }
}

# 8) delete_event
Usuario: "Elimina la reunión de revisión del miércoles"
Respuesta:
{
  "function": "delete_event",
  "parameters": {
    "name": "Reunión de revisión",
    "start_date": "2025-10-08"
  }
}

# 9) duplicate_event
Usuario: "Duplica la reunión de equipo del 3 al 10 de octubre a las 16h"
Respuesta:
{
  "function": "duplicate_event",
  "parameters": {
    "name": "Reunión de equipo",
    "original_date": "2025-10-03",
    "new_date": "2025-10-10",
    "new_time": "16:00"
  }
}

# 10) patch_event
Usuario: "Cambia el título de la reunión del 6 de octubre a 'Entrega final'"
Respuesta:
{
  "function": "patch_event",
  "parameters": {
    "name": "Reunión",
    "start_date": "2025-10-06",
    "changes": {
      "summary": "Entrega final"
    }
  }
}

# 11) get_events — consultas por periodo o nombre
Usuario: "Dime los eventos de esta semana"
Respuesta:
{
  "function": "get_events",
  "parameters": {
    "start_date": "2025-10-03",
    "end_date": "2025-10-10"
  }
}

Usuario: "Muéstrame los eventos de este mes"
Respuesta:
{
  "function": "get_events",
  "parameters": {
    "start_date": "2025-10-01",
    "end_date": "2025-10-31"
  }
}

Usuario: "Enséñame los eventos de clase de inglés"
Respuesta:
{
  "function": "get_events",
  "parameters": {
    "name": "Clase de inglés"
  }
}

Usuario: "Lista los eventos de la semana del 7 al 13 de octubre"
Respuesta:
{
  "function": "get_events",
  "parameters": {
    "start_date": "2025-10-07",
    "end_date": "2025-10-13"
  }
}
"""


prompt = f"""
⚠️ IMPORTANTE:
Hoy es {fecha_actual}.
Usa esta fecha como referencia para interpretar expresiones relativas
como "hoy", "mañana", "el viernes", etc.
Debes usar la fecha real del sistema en el momento de ejecución, no la de los ejemplos.

{p}
"""



In [151]:
response = model.generate_content(prompt)


In [16]:
import json

test_phrases = [
    # ---- create_event ----
    "Pon una reunión con Marta mañana a las 10 en la biblioteca",
    "Agenda una cita médica el 10 de octubre a las 9:00 en Madrid",
    "Organiza clase de repaso el viernes todo el día",
    "Programa tutoría con Pedro los lunes a las 12 durante 4 semanas",
    "Añade evento: presentación de TFG el 15 de noviembre a las 17:00, invita a laura@email.com y juan@email.com",
    "Crea reunión de investigación el 20 de octubre a las 15h, color azul",
    "Pon una reunión de equipo el 25 de octubre a las 10h, invita a Ana",
    "Agrega cita con dentista el 30 de octubre de 10:00 a 11:00",
    "Programa práctica de laboratorio el 5 de noviembre todo el día",
    "Agenda conferencia online el 12 de noviembre a las 18h con enlace Meet",

    # ---- delete_event ----
    "Elimina la reunión con Juan del 6 de octubre",
    "Quita la clase de inglés del 5 al 8 de octubre",
    "Borra la cita médica del 10 de octubre",
    "Suprime evento Trabajo grupal",
    "Cancela la tutoría del martes",
    "Elimina la presentación de TFG del 15 de noviembre",
    "Quita la reunión de investigación del 20",
    "Borra la conferencia online del 12 de noviembre",
    "Suprime el taller de laboratorio del 5 de noviembre",
    "Elimina la reunión semanal con Pedro del lunes",

    # ---- duplicate_event ----
    "Duplica la reunión de equipo del 3 al 10 de octubre a las 16h",
    "Copia el examen del 8 de octubre al 15, misma hora",
    "Clona la reunión de investigación del 4 al 11 de octubre",
    "Repite la tutoría del 7 al 14 de octubre a las 9h",
    "Duplica cita con Ana del lunes al miércoles",
    "Copia el taller de laboratorio del 5 al 12 de noviembre",
    "Clona la conferencia online del 12 al 19 de noviembre",
    "Repite la reunión de ventas del 6 al 13 de octubre",
    "Duplica la clase de repaso del viernes al siguiente viernes",
    "Copia la reunión con Marta del 10 al 20 de octubre",

    # ---- patch_event ----
    "Cambia el título de la reunión del 6 de octubre a 'Entrega final'",
    "Modifica la reunión con Ana del 7: pon ubicación Aula 3.1",
    "Actualiza la reunión con Juan del 8: cámbiala a las 12:00-13:30",
    "Edita la cita médica del 9: márcala en rojo",
    "Ajusta la tutoría del viernes: cambia descripción a 'Revisión parcial del TFG'",
    "Modifica el examen del 15: cambia hora a 17:00-18:30",
    "Edita la reunión de investigación del 20: pon enlace Meet",
    "Cambia la conferencia online del 12: color amarillo",
    "Actualiza el taller del 5: cambia ubicación a Laboratorio B",
    "Modifica la reunión semanal con Pedro: ajusta hora a 11:00",

    # ---- get_events ----
    "Dime los eventos de esta semana.",
    "Muéstrame todos mis eventos de octubre.",
    "Enséñame las clases de inglés programadas este mes.",
    "Lista las reuniones con Pedro de la próxima semana.",
    "Quiero ver los eventos del calendario de trabajo entre el 10 y el 20 de octubre.",
    "Consulta los eventos del 15 al 18 de noviembre.",
    "Búscame las tutorías de este mes.",
    "Obtén todos los eventos que tengo la semana que viene.",
    "Dime qué eventos hay mañana.",
    "Muestra los eventos relacionados con el TFG de octubre."
]


for phrase in test_phrases:
    full_prompt = f"{prompt}\n\nUsuario: {phrase}\n"
    resp = model.generate_content(full_prompt)
    print("Usuario:", phrase)
    print("Gemini →", resp.text, "\n")


Usuario: Pon una reunión con Marta mañana a las 10 en la biblioteca
Gemini → ```json
{
  "function": "create_event",
  "parameters": {
    "name": "Reunión con Marta",
    "start_date": "2025-10-09",
    "start_time": "10:00",
    "location": "Biblioteca"
  }
}
``` 

Usuario: Agenda una cita médica el 10 de octubre a las 9:00 en Madrid
Gemini → ```json
{
  "function": "create_event",
  "parameters": {
    "name": "Cita médica",
    "start_date": "2025-10-10",
    "start_time": "09:00",
    "location": "Madrid"
  }
}
``` 

Usuario: Organiza clase de repaso el viernes todo el día
Gemini → ```json
{
  "function": "create_event",
  "parameters": {
    "name": "Clase de repaso",
    "start_date": "2025-10-10"
  }
}
``` 

Usuario: Programa tutoría con Pedro los lunes a las 12 durante 4 semanas
Gemini → ```json
{
  "function": "create_event",
  "parameters": {
    "name": "Tutoría con Pedro",
    "start_date": "2025-10-13",
    "start_time": "12:00",
    "recurrence": [
      "RRULE:FREQ=WEEK

In [ ]:
import json
    
# Ruta segura al directorio 'tests'
base_path = os.path.join(os.getcwd(), "tests")
os.makedirs(base_path, exist_ok=True)

# Archivos de entrada/salida
inputs_path = os.path.join(base_path, "gemini_inputs.jsonl")
outputs_path = os.path.join(base_path, "gemini_outputs.jsonl")    

# Escritura de archivos
with open(inputs_path, "w", encoding="utf-8") as fin, \
     open(outputs_path, "w", encoding="utf-8") as fout:
    for phrase in test_phrases:
        full_prompt = f"{prompt}\n\nUsuario: {phrase}\n"
        resp = model.generate_content(full_prompt)
        
        fin.write(json.dumps({"input": phrase}, ensure_ascii=False) + "\n")
        fout.write(json.dumps({"output": resp.text}, ensure_ascii=False) + "\n")

print("✅ Pruebas guardadas en tests/gemini_inputs.jsonl y tests/gemini_outputs.jsonl")


In [35]:
import json
import re
import inspect

FUNCTION_MAP = {
    "create_event": create_event,
    "delete_event": delete_event,
    "duplicate_event": duplicate_event,
    "patch_event": patch_event,
    "get_events": get_events
}

def extract_json_from_output(output_str: str) -> dict:
    """Limpia el bloque ```json ... ``` y lo convierte en dict."""
    if not output_str:
        return {}
    cleaned = re.sub(r"```json|```", "", output_str, flags=re.IGNORECASE).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print("⚠️ No se pudo interpretar el JSON:\n", cleaned)
        return {}

def comprobar_gemini_respuestas(file_path: str):
    """Comprueba que las respuestas JSON de Gemini son válidas según tus funciones."""
    print("🔍 Iniciando comprobación de salidas JSON...\n")

    with open(file_path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                data = json.loads(line)
                output = data.get("output", "")
                obj = extract_json_from_output(output)

                fn_name = obj.get("function")
                params = obj.get("parameters", {})

                print(f"\n🧩 Test #{line_number}: {fn_name}")
                print(json.dumps(params, indent=2, ensure_ascii=False))

                if fn_name not in FUNCTION_MAP:
                    print(f"❌ Función '{fn_name}' no existe en FUNCTION_MAP.")
                    continue

                fn = FUNCTION_MAP[fn_name]
                sig = inspect.signature(fn)
                valid_args = list(sig.parameters.keys())
                params_json = list(params.keys())

                extra = [p for p in params_json if p not in valid_args]
                faltan = [
                    p for p, v in sig.parameters.items()
                    if v.default == inspect._empty
                    and p not in params_json
                    and p not in ("svc", "service")
                ]

                if extra:
                    print(f"⚠️ Parámetros no reconocidos: {extra}")
                if faltan:
                    print(f"⚠️ Faltan parámetros obligatorios: {faltan}")
                if not extra and not faltan:
                    print("✅ JSON válido y coincide con la función.")

            except Exception as e:
                print(f"❌ Error en línea {line_number}: {e}")

    print("\n🏁 Comprobación finalizada.")


In [38]:
comprobar_gemini_respuestas("tests/gemini_outputs.jsonl")

🔍 Iniciando comprobación de salidas JSON...


🧩 Test #1: create_event
{
  "name": "Reunión con Marta",
  "start_date": "2025-10-09",
  "start_time": "10:00",
  "location": "Biblioteca"
}
✅ JSON válido y coincide con la función.

🧩 Test #2: create_event
{
  "name": "Cita médica",
  "start_date": "2025-10-10",
  "start_time": "09:00",
  "location": "Madrid"
}
✅ JSON válido y coincide con la función.

🧩 Test #3: create_event
{
  "name": "Clase de repaso",
  "start_date": "2025-10-10"
}
✅ JSON válido y coincide con la función.

🧩 Test #4: create_event
{
  "name": "Tutoría con Pedro",
  "start_date": "2025-10-13",
  "start_time": "12:00",
  "end_time": "13:00",
  "recurrence": [
    "RRULE:FREQ=WEEKLY;BYDAY=MO;COUNT=4"
  ]
}
✅ JSON válido y coincide con la función.

🧩 Test #5: create_event
{
  "name": "Presentación de TFG",
  "start_date": "2025-11-15",
  "end_date": "2025-11-15",
  "start_time": "17:00",
  "end_time": "18:00",
  "attendees": [
    "laura@email.com",
    "juan@email.com"
  